In [62]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import pickle
from tqdm import tqdm

# ====== CONFIG ======
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = "data/train"
SSL_ENCODER_PATH = "ssl_encoder.pth"
FSL_ENCODER_PATH = "fsl_encoder.pth"
SAVE_PATH = "hybrid_model.pth"
PKL_PATH = "hybrid_features.pkl"
EPOCHS = 30
BATCH_SIZE = 8
LR = 1e-4
NUM_CLASSES = 2  # adjust to your dataset

# ====== DATASET ======
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_data = datasets.ImageFolder(root=DATA_ROOT, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

print(f"📂 Loaded dataset with {len(train_data)} samples across {len(train_data.classes)} classes.")

# ====== LOAD SSL + FSL ENCODERS ======
def load_encoder(path):
    encoder = models.resnet18(weights=None)
    encoder = nn.Sequential(*list(encoder.children())[:-1])
    state = torch.load(path, map_location=DEVICE)
    try:
        encoder.load_state_dict(state, strict=False)
        print(f"✅ Loaded encoder weights from {path}")
    except Exception as e:
        print(f"⚠️ Warning: Partial state load for {path}: {e}")
    return encoder.to(DEVICE)

ssl_encoder = load_encoder(SSL_ENCODER_PATH)
fsl_encoder = load_encoder(FSL_ENCODER_PATH)

# ====== HYBRID MODEL ======
class HybridModel(nn.Module):
    def __init__(self, ssl_encoder, fsl_encoder, num_classes):
        super().__init__()
        self.ssl_encoder = ssl_encoder
        self.fsl_encoder = fsl_encoder
        self.fc = nn.Sequential(
            nn.Linear(512 * 2, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        with torch.no_grad():
            ssl_feat = self.ssl_encoder(x).flatten(1)
            fsl_feat = self.fsl_encoder(x).flatten(1)
        fused = torch.cat([ssl_feat, fsl_feat], dim=1)
        out = self.fc(fused)
        return out, fused

hybrid_model = HybridModel(ssl_encoder, fsl_encoder, len(train_data.classes)).to(DEVICE)
optimizer = optim.Adam(hybrid_model.fc.parameters(), lr=LR)

# ====== TRAIN HYBRID MODEL ======
for epoch in range(EPOCHS):
    hybrid_model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs, _ = hybrid_model(imgs)
        loss = F.cross_entropy(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc*100:.2f}%")

torch.save(hybrid_model.state_dict(), SAVE_PATH)
print(f"✅ Hybrid model saved as {SAVE_PATH}")

# ====== SAVE FEATURES + METRICS ======
hybrid_model.eval()
all_feats, all_labels, all_preds = [], [], []
with torch.no_grad():
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE)
        outputs, feats = hybrid_model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        all_feats.append(feats.cpu().numpy())
        all_labels.append(labels.numpy())
        all_preds.append(preds)

all_feats = np.concatenate(all_feats)
all_labels = np.concatenate(all_labels)
all_preds = np.concatenate(all_preds)
acc = (all_preds == all_labels).mean()

hybrid_results = {
    "features": all_feats,
    "labels": all_labels,
    "predictions": all_preds,
    "accuracy": acc,
}
with open(PKL_PATH, "wb") as f:
    pickle.dump(hybrid_results, f)
print(f"✅ Hybrid features and metrics saved to {PKL_PATH} (Acc: {acc*100:.2f}%)")


📂 Loaded dataset with 1000 samples across 2 classes.
✅ Loaded encoder weights from ssl_encoder.pth
✅ Loaded encoder weights from fsl_encoder.pth


Epoch 1/30:   2%|▏         | 2/125 [00:21<21:36, 10.54s/it]


KeyboardInterrupt: 

In [63]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import numpy as np
import pickle
from tqdm import tqdm

# ====== CONFIG ======
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = "data/train"
SSL_ENCODER_PATH = "ssl_encoder.pth"
FSL_ENCODER_PATH = "fsl_encoder.pth"
SAVE_PATH = "hybrid_model.pth"
PKL_PATH = "hybrid_features.pkl"
EPOCHS = 30
BATCH_SIZE = 4
LR_FC = 1e-4
LR_ENCODER = 1e-5
NUM_CLASSES = 2  # adjust to your dataset
VAL_SPLIT = 0.2

# ====== DATASET & TRANSFORMS ======
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_ROOT, transform=transform)
val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"📂 Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
print(f"Classes: {full_dataset.classes}")

# ====== LOAD SSL + FSL ENCODERS ======
def load_encoder(path):
    encoder = models.resnet18(weights=None)
    encoder = nn.Sequential(*list(encoder.children())[:-1])  # remove fc
    state = torch.load(path, map_location=DEVICE)
    encoder.load_state_dict(state, strict=False)
    return encoder.to(DEVICE)

ssl_encoder = load_encoder(SSL_ENCODER_PATH)
fsl_encoder = load_encoder(FSL_ENCODER_PATH)

# Unfreeze last layer of encoders for fine-tuning
for param in ssl_encoder[-2:].parameters():
    param.requires_grad = True
for param in fsl_encoder[-2:].parameters():
    param.requires_grad = True

# ====== HYBRID MODEL ======
class HybridModel(nn.Module):
    def __init__(self, ssl_encoder, fsl_encoder, num_classes):
        super().__init__()
        self.ssl_encoder = ssl_encoder
        self.fsl_encoder = fsl_encoder
        self.fc = nn.Sequential(
            nn.Linear(512*2, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        ssl_feat = self.ssl_encoder(x).flatten(1)
        fsl_feat = self.fsl_encoder(x).flatten(1)
        fused = torch.cat([ssl_feat, fsl_feat], dim=1)
        out = self.fc(fused)
        return out, fused

hybrid_model = HybridModel(ssl_encoder, fsl_encoder, NUM_CLASSES).to(DEVICE)

# ====== OPTIMIZER ======
optimizer = optim.Adam([
    {"params": hybrid_model.fc.parameters(), "lr": LR_FC},
    {"params": hybrid_model.ssl_encoder.parameters(), "lr": LR_ENCODER},
    {"params": hybrid_model.fsl_encoder.parameters(), "lr": LR_ENCODER}
])

# ====== TRAINING LOOP ======
for epoch in range(EPOCHS):
    hybrid_model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs, _ = hybrid_model(imgs)
        loss = F.cross_entropy(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # Validation
    hybrid_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs, _ = hybrid_model(imgs)
            preds = outputs.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")

# ====== SAVE MODEL ======
torch.save(hybrid_model.state_dict(), SAVE_PATH)
print(f"✅ Hybrid model saved as {SAVE_PATH}")

# ====== SAVE FEATURES & METRICS ======
hybrid_model.eval()
all_feats, all_labels, all_preds = [], [], []
with torch.no_grad():
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE)
        outputs, feats = hybrid_model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        all_feats.append(feats.cpu().numpy())
        all_labels.append(labels.numpy())
        all_preds.append(preds)

all_feats = np.concatenate(all_feats)
all_labels = np.concatenate(all_labels)
all_preds = np.concatenate(all_preds)
acc = (all_preds == all_labels).mean()

hybrid_results = {
    "features": all_feats,
    "labels": all_labels,
    "predictions": all_preds,
    "accuracy": acc
}

with open(PKL_PATH, "wb") as f:
    pickle.dump(hybrid_results, f)
print(f"✅ Hybrid features and metrics saved to {PKL_PATH} (Train Acc: {acc*100:.2f}%)")


📂 Train samples: 800, Val samples: 200
Classes: ['fake', 'real']


Epoch 1/30:  13%|█▎        | 26/200 [06:07<40:57, 14.12s/it]


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import numpy as np
import pickle
from tqdm import tqdm

# ====== CONFIG ======
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = "data/train"
SSL_ENCODER_PATH = "ssl_encoder.pth"
FSL_ENCODER_PATH = "fsl_encoder.pth"
SAVE_PATH = "hybrid_model.pth"
PKL_PATH = "hybrid_features.pkl"
EPOCHS = 50
BATCH_SIZE = 4  # large images → small batch to avoid OOM
LR_FC = 1e-4
LR_ENCODER = 1e-5
NUM_CLASSES = 2
VAL_SPLIT = 0.2

# ====== DATASET & TRANSFORMS ======
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_ROOT, transform=transform)
val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"📂 Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
print(f"Classes: {full_dataset.classes}")

# ====== LOAD ENCODERS ======
def load_encoder(path):
    encoder = models.resnet18(weights=None)
    encoder = nn.Sequential(*list(encoder.children())[:-1])  # remove fc
    state = torch.load(path, map_location=DEVICE)
    encoder.load_state_dict(state, strict=False)
    return encoder.to(DEVICE)

ssl_encoder = load_encoder(SSL_ENCODER_PATH)
fsl_encoder = load_encoder(FSL_ENCODER_PATH)

# Unfreeze last residual block only
for param in ssl_encoder[-2:].parameters():
    param.requires_grad = True
for param in fsl_encoder[-2:].parameters():
    param.requires_grad = True

# ====== HYBRID MODEL ======
class HybridModel(nn.Module):
    def __init__(self, ssl_encoder, fsl_encoder, num_classes):
        super().__init__()
        self.ssl_encoder = ssl_encoder
        self.fsl_encoder = fsl_encoder
        self.fc = nn.Sequential(
            nn.Linear(512*2, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        ssl_feat = self.ssl_encoder(x).flatten(1)
        fsl_feat = self.fsl_encoder(x).flatten(1)
        fused = torch.cat([ssl_feat, fsl_feat], dim=1)
        out = self.fc(fused)
        return out, fused

hybrid_model = HybridModel(ssl_encoder, fsl_encoder, NUM_CLASSES).to(DEVICE)

# ====== OPTIMIZER & SCHEDULER ======
optimizer = optim.Adam([
    {"params": hybrid_model.fc.parameters(), "lr": LR_FC},
    {"params": hybrid_model.ssl_encoder.parameters(), "lr": LR_ENCODER},
    {"params": hybrid_model.fsl_encoder.parameters(), "lr": LR_ENCODER}
])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)


# ====== TRAINING LOOP ======
best_val_acc = 0.0

for epoch in range(EPOCHS):
    hybrid_model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs, _ = hybrid_model(imgs)
        loss = F.cross_entropy(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # Validation
    hybrid_model.eval()
    val_correct, val_total, val_loss = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs, _ = hybrid_model(imgs)
            loss = F.cross_entropy(outputs, labels)
            val_loss += loss.item()
            preds = outputs.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total
    val_loss /= len(val_loader)

    scheduler.step(val_acc)  # reduce LR if val_acc plateaus

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Val Loss: {val_loss:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(hybrid_model.state_dict(), SAVE_PATH)
        print(f"✅ Saved best model at epoch {epoch+1} with Val Acc: {best_val_acc*100:.2f}%")

# ====== SAVE FEATURES & METRICS ======
hybrid_model.eval()
all_feats, all_labels, all_preds = [], [], []
with torch.no_grad():
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE)
        outputs, feats = hybrid_model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        all_feats.append(feats.cpu().numpy())
        all_labels.append(labels.numpy())
        all_preds.append(preds)

all_feats = np.concatenate(all_feats)
all_labels = np.concatenate(all_labels)
all_preds = np.concatenate(all_preds)
acc = (all_preds == all_labels).mean()

hybrid_results = {
    "features": all_feats,
    "labels": all_labels,
    "predictions": all_preds,
    "accuracy": acc
}

with open(PKL_PATH, "wb") as f:
    pickle.dump(hybrid_results, f)
print(f"✅ Hybrid features and metrics saved to {PKL_PATH} (Train Acc: {acc*100:.2f}%)")


📂 Train samples: 800, Val samples: 200
Classes: ['fake', 'real']


Epoch 1/50:  37%|███▋      | 74/200 [15:47<28:47, 13.71s/it]